# Pi5 Thermal Study — Analysis Notebook

Loads all 18 experiment CSVs, builds the summary table, and renders figures
for the paper. This notebook mirrors `src/analyze.py` but lets you explore
interactively and tweak individual plots before exporting final paper figures.


In [1]:
import sys
sys.path.insert(0, "../src")
from analyze import (
    discover_runs, load_power_log, build_summary, save_tables,
    load_full_system_df, load_full_inference_df,
    plot_temp_over_time, plot_fps_by_format, plot_inference_time_box,
    plot_passive_vs_active_temp_rise, plot_power_by_format,
)
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline


## 1. Discover and load all runs

In [2]:
runs = discover_runs()
power_df = load_power_log()
summary = build_summary(runs, power_df)
summary


[info] Found 18 complete run(s) out of 18 expected.


,model,format,cooling,run_stem,n_frames,duration_min,inference_ms_mean,inference_ms_min,inference_ms_max,fps_mean,...,temp_end_C,temp_max_C,temp_rise_C,cpu_freq_min_MHz,throttled,ram_mean_MB,cpu_percent_mean,watts_t1,watts_t45,watts_avg
0,yolov8n,pytorch,passive,20260624_203100,200,4.1,704.3,682.5,727.7,3.84,...,59.8,59.8,15.8,2400.0,no,790.7,46.1,7.61,7.61,7.61
1,yolov8n,onnx,passive,20260624_203100,200,4.1,505.1,482.5,527.3,3.86,...,60.4,60.4,15.5,2400.0,no,789.6,45.8,10.50,11.20,10.85
2,yolov8n,openvino,passive,20260624_203100,200,4.1,905.5,883.5,929.0,3.85,...,59.6,59.6,15.6,2400.0,no,789.4,46.3,11.20,11.20,11.20
3,yolo11n,pytorch,passive,20260624_203100,200,4.1,704.8,680.9,727.7,3.85,...,59.4,60.2,14.5,2400.0,no,790.3,45.8,11.20,11.20,11.20
4,yolo11n,onnx,passive,20260624_203100,200,4.1,505.2,482.7,528.2,3.85,...,59.9,59.9,15.7,2400.0,no,790.1,46.0,10.10,10.60,10.35
5,yolo11n,openvino,passive,20260624_203100,200,4.1,905.3,882.9,928.3,3.86,...,60.0,60.0,15.8,2400.0,no,789.0,45.7,10.60,10.60,10.60
6,yolo12n,pytorch,passive,20260624_203100,200,4.1,705.3,681.4,729.0,3.85,...,60.2,60.2,15.9,2400.0,no,791.0,45.7,7.27,7.39,7.33
7,yolo12n,onnx,passive,20260624_203100,200,4.1,505.9,482.7,526.8,3.84,...,59.9,59.9,15.1,2400.0,no,790.5,46.1,10.30,10.70,10.50
8,yolo12n,openvino,passive,20260624_203100,200,4.1,904.0,881.5,927.2,3.84,...,60.3,60.3,15.7,2400.0,no,788.1,45.9,9.06,9.89,9.48
9,yolov8n,pytorch,active,20260624_203100,200,4.1,700.3,680.7,719.0,3.84,...,47.0,47.6,2.2,2400.0,no,790.1,45.8,6.13,6.75,6.44


## 2. Save tables (CSV + Markdown for paper)

In [3]:
save_tables(summary)

[info] Saved /home/claude/analysis_dev/results/tables/summary_table.csv
[info] Saved /home/claude/analysis_dev/results/tables/summary_table.md


## 3. Load full per-frame and per-sample data

In [4]:
sysdf = load_full_system_df(runs)
infdf = load_full_inference_df(runs)
sysdf.head()


,timestamp,elapsed_seconds,temperature_C,cpu_freq_MHz,throttle_status,fps,ram_usage_MB,cpu_percent,model,format,cooling,elapsed_min
0,2026-06-24 20:31:00,0.0,45.7,2400.0,throttled=0x0,3.78,799.0,44.5,yolo11n,onnx,active,0.000000
1,2026-06-24 20:31:05,5.0,44.5,2400.0,throttled=0x0,3.76,799.6,47.6,yolo11n,onnx,active,0.083333
2,2026-06-24 20:31:10,10.0,45.4,2400.0,throttled=0x0,3.80,780.4,44.3,yolo11n,onnx,active,0.166667
3,2026-06-24 20:31:15,15.0,44.9,2400.0,throttled=0x0,3.78,787.2,46.5,yolo11n,onnx,active,0.250000
4,2026-06-24 20:31:20,20.0,45.2,2400.0,throttled=0x0,3.91,793.3,44.9,yolo11n,onnx,active,0.333333


## 4. Key figure — Temperature over time (passive vs active)

In [5]:
plot_temp_over_time(sysdf)

[info] Saved /home/claude/analysis_dev/results/figures/temperature_over_time.png


## 5. FPS by model/format/cooling

In [6]:
plot_fps_by_format(summary)

[info] Saved /home/claude/analysis_dev/results/figures/fps_by_format.png


## 6. Inference time distributions

In [7]:
plot_inference_time_box(infdf)

[info] Saved /home/claude/analysis_dev/results/figures/inference_time_distribution.png


## 7. Temperature rise: passive vs active (headline thermal finding)

In [8]:
plot_passive_vs_active_temp_rise(summary)

[info] Saved /home/claude/analysis_dev/results/figures/temp_rise_passive_vs_active.png


## 8. Power draw by format

In [9]:
plot_power_by_format(summary)

[info] Saved /home/claude/analysis_dev/results/figures/power_by_format.png


## 9. Quick stats for the paper text

Use the cells below to pull specific numbers you'll quote directly in the
manuscript (e.g. "OpenVINO was X% slower than PyTorch on Pi 5").

In [10]:
# OpenVINO vs PyTorch slowdown, passive condition
pivot = summary.pivot_table(index=["model","cooling"], columns="format", values="inference_ms_mean")
pivot["openvino_vs_pytorch_pct"] = ((pivot["openvino"] - pivot["pytorch"]) / pivot["pytorch"] * 100).round(1)
pivot


format           pytorch   onnx  openvino  openvino_vs_pytorch_pct
model   cooling                                                   
yolov8n passive    704.3  505.1     905.5                     28.6
        active     700.3  499.9     899.5                     28.4
yolo11n passive    704.8  505.2     905.3                     28.4
        active     700.1  500.2     898.9                     28.4
yolo12n passive    705.3  505.9     904.0                     28.2
        active     699.8  499.5     899.8                     28.6

In [11]:
# Average temperature rise: passive vs active
summary.groupby("cooling")["temp_rise_C"].agg(["mean","min","max"]).round(1)


,mean,min,max
cooling,,,
passive,15.5,14.5,15.9
active,2.5,1.9,3.2
